In [1]:
import openfhe_numpy as onp
from openfhe import *
import numpy as np

In [3]:
def decrypt_and_print(entity, keys):
    decrypted = entity.decrypt(keys.secretKey, unpack_type="original")
    print(decrypted)

In [4]:
mult_depth = 10

params = CCParamsCKKSRNS()
params.SetMultiplicativeDepth(mult_depth)
params.SetScalingModSize(59)
params.SetFirstModSize(60)
params.SetScalingTechnique(FIXEDAUTO)
params.SetKeySwitchTechnique(HYBRID)
params.SetSecretKeyDist(UNIFORM_TERNARY)

cc = GenCryptoContext(params)
cc.Enable(PKESchemeFeature.PKE)
cc.Enable(PKESchemeFeature.LEVELEDSHE)
cc.Enable(PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()
cc.EvalMultKeyGen(keys.secretKey)
cc.EvalSumKeyGen(keys.secretKey)

batch_size = cc.GetRingDimension() // 2
print("\n****** CRYPTO PARAMETERS ******")
print(f"Total Slots: {batch_size}")
print("*******************************")


****** CRYPTO PARAMETERS ******
Total Slots: 32768
*******************************


In [15]:
print("\nInput")
print("\nMatrix:\n", matrix, matrix.shape)
print("\nVector:\n", vector, vector.shape)

print("\n" + "*" * 60)
print(f"* Homomorphic Matrix Vector Product")
print("*" * 60)

expected = matrix @ vector
print(f"\nExpected:\n{expected}")


Input

Matrix:
 [[0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]] (2, 5)

Vector:
 [1. 2. 3. 4. 5.] (5,)

************************************************************
* Homomorphic Matrix Vector Product
************************************************************

Expected:
[2. 3.]


In [22]:
ctm_m_rm = onp.array(
    cc=cc,
    data=matrix,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)
ctm_m_rm.extra["colkey"] = onp.sum_col_keys(keys.secretKey)
ctv_v_cm = onp.array(
    cc=cc,
    data=vector,
    batch_size=batch_size,
    order=onp.COL_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

In [83]:
def slice_vector(enc_vec, start, end, keys):
    """
    Slice an encrypted vector while keeping it encrypted using matrix multiplication.
    
    Treat the vector of length n as a matrix of shape (1, n).
    Create a selection mask in numpy which is 0 everywhere except for the range [start, end)
    Matrix multiply the selection matrix with the original vector to get the sliced result. 
    
    Parameters
    ----------
    enc_vec :  CTArray
        The encrypted vector to slice (in ROW_MAJOR format)
    start : int
        Start index (inclusive)
    end : int
        End index (exclusive)
    keys:
        The keys generated by the cryptocontext (cc.KeyGen())
    
    Returns
    -------
    CTArray
        The sliced encrypted vector containing elements from index start to end-1
    """
    slice_length = end - start
    length = enc_vec.original_shape[0]

    selection_matrix = np.zeros((slice_length, length))
    for i in range(slice_length):
        selection_matrix[i, start + i] = 1.0

    enc_selection = onp.array(
        cc=enc_vec.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_vec.batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey) # maybe this can be moved outside so only public key would be needed in this function

    slice_result = enc_selection @ enc_vec

    return slice_result

In [78]:
ohe_matrix = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=np.float32)
ohe_matrix

array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1.]], dtype=float32)

In [45]:
start1, end1 = 0, 8
start2, end2 = 8, 12

In [46]:
cols1 = ohe_matrix[:, start1:end1]
cols2 = ohe_matrix[:, start2:end2]

In [47]:
cols1, cols2

(array([[0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]], dtype=float32))

In [49]:
cols1.shape, cols2.shape

((5, 8), (5, 4))

In [54]:
print((cols1.T @ cols2).flatten())

[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 0. 0.]


In [66]:
def select(matrix, start, end):
    n_cols = matrix.shape[-1]
    n_selected = end - start
    
    # Create selection matrix: (n_cols, n_selected)
    # Each column j has a 1 at position (start + j)
    selection_matrix = np.zeros((n_cols, n_selected))
    for i in range(n_selected):
        selection_matrix[start + i, i] = 1.0
    
    # Matrix multiplication: (rows, n_cols) @ (n_cols, n_selected) = (rows, n_selected)
    return matrix @ selection_matrix

In [67]:
select(ohe_matrix, 0, 8)

array([[0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.]])

In [79]:
enc_matrix = onp.array(
        cc=cc,
        data=ohe_matrix,
        batch_size=batch_size,
        order=onp.COL_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )

In [91]:
def select_columns(enc_matrix, start, end, keys):
    """
    Select columns from an encrypted matrix while keeping it encrypted using matrix multiplication.
    
    Parameters
    ----------
    enc_matrix : CTArray
        The encrypted matrix to select columns from (in ROW_MAJOR format)
        Shape: (rows, n_cols)
    start : int
        Start column index (inclusive)
    end : int
        End column index (exclusive)
    keys : KeyPair
        The keys generated by the cryptocontext (cc.KeyGen())
    
    Returns
    -------
    CTArray
        The matrix with selected columns, shape (rows, end-start)
    """
    n_cols = enc_matrix.original_shape[-1]
    n_selected = end - start
    
    # Create selection matrix: (n_cols, n_selected)
    # Each column j has a 1 at position (start + j)
    selection_matrix = np.zeros((n_cols, n_selected))
    for j in range(n_selected):
        selection_matrix[start + j, j] = 1.0
    
    # Encrypt selection matrix with COMPLEMENTARY encoding
    # Matrix is ROW_MAJOR, so if enc_matrix is ROW_MAJOR, use COL_MAJOR here
    enc_selection = onp.array(
        cc=enc_matrix.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_matrix.batch_size,
        order=onp.COL_MAJOR,  # Complementary to enc_matrix's ROW_MAJOR
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    
    # Attach column sum key (needed for matrix-matrix multiplication)
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey)
    
    # Matrix multiplication: (rows, n_cols) @ (n_cols, n_selected) = (rows, n_selected)
    result = enc_matrix @ enc_selection
    decrypt_and_print(result, keys)
    return result
